# ncRNA orthology oracle: reciprocal-annotation truth from real mm39 GENCODE

**Goal.** Build a *TOGA-independent* silver ground truth for ncRNA orthology by projecting each
hg38 ncRNA through the hg38→mm39 chain and asking **"is there anything *similar* in the intersection"**
of the projected locus with the real mm39 GENCODE annotation.

Because it uses an *independent* annotation (mm39 GENCODE, not produced by this pipeline or by TOGA),
agreement here is a real signal — unlike the current `model.json`, which was trained to *reproduce*
protein-coding TOGA labels.

**Label tiers per hg38 ncRNA (best over its candidate chains):**
- `NAME_MATCH`   — projected locus overlaps an mm39 ncRNA with the same gene symbol (case-insensitive: `MIR21`/`Mir21`). Near-gold.
- `BIOTYPE_MATCH`— overlaps an mm39 ncRNA of a compatible biotype group. Positive.
- `PSEUDO_HIT`   — overlaps an mm39 *pseudogene* (tests the P_PGENES question for ncRNA).
- `FAMILY_MISMATCH` — overlaps an mm39 ncRNA of a **different** family. Clean negative (ortholog ≠ homolog).
- `UNLABELED_INTERGENIC` — projects, but nothing annotated there. **Not a negative** (mm39 annotation gap).
- `NO_PROJECTION` / `NO_CHAINS` — no syntenic chain projects the gene (loss, or SPAN/divergent — see below).

**Two side-analyses fall out for free:** SPAN divergent-orthologs (exon projection fails but the *span*
lands on a compatible mm39 ncRNA → feed to RiNALMo) and pseudogene concordance.


## 1. Config — knobs for the experiment
First run defaults to `CHROM_SUBSET = ["chr19","chr21"]` (dense + small) so it finishes in minutes.
Set `CHROM_SUBSET = None` for genome-wide (heavier: the full 318 MB chain is loaded either way).


In [61]:
from pathlib import Path

ROOT = Path("/Users/Bogdan.Kirilenko/Developer/CURIA_pipeline")

REF_BED   = ROOT/"input_data/reference_annotation/hg38.input.w.tRNA.bed"
REF_META  = ROOT/"input_data/reference_annotation/hg38.transcript_metadata.tsv"
REF_NAMES = ROOT/"input_data/reference_annotation/hg38_gene_names.txt"
QRY_BED   = ROOT/"input_data/mm39_annotation_validation/mm39_gencode_all_transcripts.bed"
QRY_META  = ROOT/"input_data/mm39_annotation_validation/mm39_gencode_metadata.tsv"
CHAIN     = ROOT/"input_data/chains/hg38.mm39.allfilled.chain.gz"

# --- experiment knobs ---
CHROM_SUBSET   = ["chr19", "chr21"]   # None => genome-wide (slow)
NCRNA_BIOTYPES = {"lncRNA","miRNA","snoRNA","scaRNA","snRNA","misc_RNA","tRNA","rRNA","vault_RNA","sRNA"}
LEN_RATIO_BAND = (0.5, 2.0)  # projected locus length / ref gene length must fall in this band.
                             # THE SPAN FIX: rejects projections that walk across a chain gap
                             # (a 3 kb gene smeared over a 100 kb locus has ratio ~33 -> rejected).
                             # Recorded per row as len_ratio so you can sweep/tighten post-hoc
                             # (e.g. (0.95,1.05) for structured RNAs, looser for lncRNA).
TOP_K_CHAINS   = 15     # per gene: project through the K highest-scoring overlapping chains
MIN_OVERLAP_BP = 1      # min bp overlap to count an mm39 hit
EXON_GAP_RATIO = 1.0    # strictness for exon-block projection
SPAN_GAP_RATIO = 25.0   # looser projection, used ONLY as SPAN fallback when exons do not align
MIN_CHAIN_SCORE= None   # e.g. 5000 to prune tiny chains at load time

In [62]:
import time
import numpy as np
import pandas as pd
from pyrion import read_bed12_file, read_chain_file
from pyrion.ops import find_intersections
from pyrion.ops.chains import project_intervals_through_chain_strict

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


## 2. Metadata joins
hg38 metadata has no `gene_name`, so ref names come via `gene_id → hg38_gene_names.txt`.
mm39 metadata carries `gene_name` directly. IDs are versioned and match the BED on both sides.

In [63]:
def strip_ver(x):
    return x.split(".")[0] if isinstance(x, str) else x

# hg38 biotype + gene_id (keyed by versioned transcript id == bed col4)
ref_meta = pd.read_csv(REF_META, sep="\t", dtype=str)
ref_biotype = dict(zip(ref_meta.transcript_id, ref_meta.transcript_biotype))
ref_gene    = dict(zip(ref_meta.transcript_id, ref_meta.gene_id))

# hg38 gene_id(no version) -> gene name
gn = pd.read_csv(REF_NAMES, sep="\t", dtype=str)
id_col   = [c for c in gn.columns if c.lower().startswith("gene stable")][0]
name_col = [c for c in gn.columns if c.lower() == "gene name"][0]
gene_name_ref = dict(zip(gn[id_col], gn[name_col]))

def ref_name_of(tid):
    g = ref_gene.get(tid)
    return gene_name_ref.get(strip_ver(g)) if g else None

# mm39 biotype + name (versioned ids == bed col4)
qry_meta = pd.read_csv(QRY_META, sep="\t", dtype=str)
mm_biotype = dict(zip(qry_meta.transcript_id, qry_meta.transcript_biotype))
mm_name    = dict(zip(qry_meta.transcript_id, qry_meta.gene_name))

print(f"hg38 biotypes: {len(ref_biotype):,} | hg38 names: {len(gene_name_ref):,} | "
      f"mm39 biotypes: {len(mm_biotype):,}")


hg38 biotypes: 386,300 | hg38 names: 46,896 | mm39 biotypes: 278,326


In [64]:
PSEUDO = {"processed_pseudogene","unprocessed_pseudogene","transcribed_processed_pseudogene","transcribed_unprocessed_pseudogene","transcribed_unitary_pseudogene","unitary_pseudogene","rRNA_pseudogene","polymorphic_pseudogene","IG_V_pseudogene","IG_C_pseudogene","TR_V_pseudogene","TR_J_pseudogene"}

GROUP = {}
GROUP.update({b: "LNC" for b in ["lncRNA"]})
GROUP.update({b: "MIR" for b in ["miRNA"]})
GROUP.update({b: "SNO" for b in ["snoRNA","scaRNA"]})
GROUP.update({b: "SNR" for b in ["snRNA"]})
GROUP.update({b: "TRNA" for b in ["tRNA","Mt_tRNA"]})
GROUP.update({b: "RRNA" for b in ["rRNA","Mt_rRNA"]})
GROUP.update({b: "MISC" for b in ["misc_RNA","vault_RNA","sRNA","scRNA"]})
GROUP.update({b: "PSEUDO" for b in PSEUDO})

TIER_RANK = {"NAME_MATCH": 3, "BIOTYPE_MATCH": 2, "PSEUDO_HIT": 1, "FAMILY_MISMATCH": 0}

def _clean_name(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    return x if x not in ("", ".") else None

def similarity(ref_bt, ref_nm, mm_id):
    mm_bt = mm_biotype.get(mm_id, "NA")
    n1, n2 = _clean_name(ref_nm), _clean_name(mm_name.get(mm_id))
    if n1 and n2 and n1.casefold() == n2.casefold():
        return "NAME_MATCH"
    g1, g2 = GROUP.get(ref_bt), GROUP.get(mm_bt)
    if g1 and g2 and g1 == g2:
        return "BIOTYPE_MATCH"
    if g2 == "PSEUDO":
        return "PSEUDO_HIT"
    return "FAMILY_MISMATCH"

# 3. Load annotations + chain (the chain load is the slow part, ~318 MB)

In [65]:
t0 = time.time(); ref = read_bed12_file(str(REF_BED)); print(f"ref bed:  {len(ref):,}  ({time.time()-t0:.0f}s)")
t0 = time.time(); qry = read_bed12_file(str(QRY_BED)); print(f"qry bed:  {len(qry):,}  ({time.time()-t0:.0f}s)")
t0 = time.time()
chains = read_chain_file(str(CHAIN), min_score=MIN_CHAIN_SCORE) if MIN_CHAIN_SCORE else read_chain_file(str(CHAIN))
print(f"chains:   {len(chains):,}  ({time.time()-t0:.0f}s)")


ref bed:  386,300  (0s)
qry bed:  278,326  (0s)
chains:   5,492,909  (41s)


In [66]:
# mm39 transcript spans per query chrom
mm_index = {}
for c in qry.get_all_chromosomes():
    ts = qry.get_by_chrom(c)
    if not ts:
        continue
    mm_index[c] = (np.array([[t.start, t.end] for t in ts], dtype=np.int64), [t.id for t in ts])

# mm39 transcript length lookup (for reciprocal coverage fractions)
mm_len = {}
for c in mm_index:
    arr, ids = mm_index[c]
    for (s, e), i in zip(arr, ids):
        mm_len[i] = int(e - s)

# chains per target(hg38) chrom: sorted arrays for fast overlap + the chain objects
chain_index = {}
target_chroms = CHROM_SUBSET or chains.get_reference_chromosomes()
for c in target_chroms:
    cl = chains.get_by_target_chrom(c)
    if not cl:
        continue
    spans  = np.array([ch.t_span for ch in cl], dtype=np.int64)
    scores = np.array([ch.score for ch in cl], dtype=np.int64)
    chain_index[c] = (spans[:, 0], spans[:, 1], scores, cl)

print("chain-indexed hg38 chroms:", {c: len(chain_index[c][3]) for c in chain_index})
print("mm_len entries:", len(mm_len))

chain-indexed hg38 chroms: {'chr19': 526632, 'chr21': 52597}
mm_len entries: 278326


## 4. Project + classify each ncRNA
For each gene: take the top-K overlapping chains by score, project its exon blocks (strict).
If exons don't project, fall back to a looser **span** projection (this is how SPAN/divergent
cases still get a locus). Apply the length cap, then intersect with mm39 and take the best tier.

In [67]:
def _norm(a):
    s, e = int(a[0]), int(a[1])
    return (s, e) if s <= e else (e, s)

def project_region(t, ch):
    """Return ((q_start,q_end), mode) or (None, None). mode in {'exon','span'}."""
    proj = project_intervals_through_chain_strict(t.blocks, ch.blocks, ch.q_strand, EXON_GAP_RATIO)
    hits = [p[0] for p in proj if not (p[0][0] == 0 and p[0][1] == 0)]
    if hits:
        a = np.array(hits)
        return (int(a[:, 0].min()), int(a[:, 1].max())), "exon"
    ps = project_intervals_through_chain_strict(
        np.array([[t.start, t.end]], dtype=np.int64), ch.blocks, ch.q_strand, SPAN_GAP_RATIO)
    p0 = ps[0][0]
    if p0[0] == 0 and p0[1] == 0:
        return None, None
    return _norm(p0), "span"

POS_TIERS = {"NAME_MATCH", "BIOTYPE_MATCH"}

def classify_transcript(t):
    bt = ref_biotype.get(t.id, "NA"); nm = ref_name_of(t.id); glen = int(t.end - t.start)
    rec = dict(transcript_id=t.id, biotype=bt, gene_name=nm, chrom=t.chrom, gene_len=glen,
               n_cand_chains=0, label="NO_CHAINS", proj_mode=None, evidence_mm=None,
               evidence_biotype=None, evidence_name=None, overlap_bp=0, region_len=None,
               len_ratio=None, cover_frac_region=None, cover_frac_mm=None)
    idx = chain_index.get(t.chrom)
    if idx is None:
        return rec
    cs, ce, sc, cl = idx
    m = (cs < t.end) & (ce > t.start)
    if not m.any():
        return rec
    cand = np.where(m)[0]; cand = cand[np.argsort(-sc[cand])][:TOP_K_CHAINS]
    rec["n_cand_chains"] = int(len(cand))
    lo, hi = LEN_RATIO_BAND
    any_proj = False; any_band = False
    best_exon = None; best_span = None   # (rank, tier, mm_id, bp, region_len, ratio)
    for i in cand:
        ch = cl[int(i)]
        region, mode = project_region(t, ch)
        if region is None:
            continue
        any_proj = True
        rlen = region[1] - region[0]
        if rlen <= 0:
            continue
        ratio = rlen / max(1, glen)
        if not (lo <= ratio <= hi):        # the length-ratio sanity band (SPAN fix)
            continue
        any_band = True
        qidx = mm_index.get(ch.q_chrom)
        if qidx is None:
            continue
        q_arr, q_ids = qidx
        ov = find_intersections(np.array([list(region)], dtype=np.int64), q_arr, [0], q_ids)
        for mm_id, bp in ov.get(0, []):
            if bp < MIN_OVERLAP_BP:
                continue
            tier = similarity(bt, nm, mm_id); r = TIER_RANK[tier]
            ev = (r, tier, mm_id, int(bp), rlen, ratio)
            if mode == "exon":
                if best_exon is None or r > best_exon[0] or (r == best_exon[0] and bp > best_exon[3]):
                    best_exon = ev
            else:
                if best_span is None or r > best_span[0] or (r == best_span[0] and bp > best_span[3]):
                    best_span = ev
    def fill(ev, mode, label):
        r, tier, mm_id, bp, rlen, ratio = ev
        mmlen = mm_len.get(mm_id, 0)
        rec.update(label=label, proj_mode=mode, evidence_mm=mm_id,
                   evidence_biotype=mm_biotype.get(mm_id), evidence_name=mm_name.get(mm_id),
                   overlap_bp=bp, region_len=int(rlen), len_ratio=round(ratio, 3),
                   cover_frac_region=round(bp / max(1, rlen), 3),
                   cover_frac_mm=round(bp / max(1, mmlen), 3) if mmlen else None)
    if best_exon is not None:
        fill(best_exon, "exon", best_exon[1])                 # NAME/BIOTYPE/PSEUDO/FAMILY_MISMATCH
    elif best_span is not None and best_span[1] in POS_TIERS:
        fill(best_span, "span", "SPAN_CANDIDATE")             # divergent-ortholog candidate -> RiNALMo
    elif best_span is not None:
        fill(best_span, "span", best_span[1])
    elif any_band:
        rec["label"] = "UNLABELED_INTERGENIC"
    elif any_proj:
        rec["label"] = "GEOM_REJECT"                         # projected, but only across a chain gap
    else:
        rec["label"] = "NO_PROJECTION"
    return rec

In [68]:
targets = [t for c in chain_index for t in ref.get_by_chrom(c)
           if ref_biotype.get(t.id) in NCRNA_BIOTYPES]
print(f"ncRNA transcripts to classify on {list(chain_index)}: {len(targets):,}")

t0 = time.time(); rows = []
for k, t in enumerate(targets):
    rows.append(classify_transcript(t))
    if k and k % 2000 == 0:
        print(f"  {k}/{len(targets)}  ({time.time()-t0:.0f}s)")
res = pd.DataFrame(rows)
print(f"done: {len(res):,} rows in {time.time()-t0:.0f}s")
res.head()


ncRNA transcripts to classify on ['chr19', 'chr21']: 9,377
  2000/9377  (2s)
  4000/9377  (4s)
  6000/9377  (6s)
  8000/9377  (8s)
done: 9,377 rows in 9s


,transcript_id,biotype,gene_name,chrom,gene_len,n_cand_chains,label,proj_mode,evidence_mm,evidence_biotype,evidence_name,overlap_bp,region_len,len_ratio,cover_frac_region,cover_frac_mm
0,ENST00000632506.1,lncRNA,NaN,chr19,10026,5,UNLABELED_INTERGENIC,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,ENST00000633703.1,lncRNA,NaN,chr19,7131,2,GEOM_REJECT,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,ENST00000816399.1,lncRNA,NaN,chr19,6194,2,GEOM_REJECT,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,ENST00000816398.1,lncRNA,NaN,chr19,6199,2,GEOM_REJECT,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,ENST00000634023.2,lncRNA,NaN,chr19,7537,3,GEOM_REJECT,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


## 5. Summary — the numbers you were missing

In [69]:
def coarse(l):
    if l in ("NAME_MATCH", "BIOTYPE_MATCH"): return "POSITIVE"        # silver positives (exon-mode only now)
    if l == "SPAN_CANDIDATE":                return "SPAN_CAND"       # divergent-ortholog candidates (RiNALMo)
    if l == "PSEUDO_HIT":                    return "PSEUDO_HIT"
    if l == "FAMILY_MISMATCH":               return "CLEAN_NEGATIVE"
    if l == "UNLABELED_INTERGENIC":          return "UNLABELED"
    if l == "GEOM_REJECT":                   return "GEOM_REJECT"
    return "NO_PROJECTION"                                            # incl. NO_CHAINS

res["class"] = res["label"].map(coarse)
piv = res.pivot_table(index="biotype", columns="class", values="transcript_id",
                      aggfunc="count", fill_value=0)
piv["TOTAL"] = piv.sum(axis=1)
piv

class,CLEAN_NEGATIVE,GEOM_REJECT,NO_PROJECTION,POSITIVE,PSEUDO_HIT,SPAN_CAND,UNLABELED,TOTAL
biotype,,,,,,,,
lncRNA,1287,3378,165,2127,308,1347,404,9016
miRNA,60,19,59,25,2,0,5,170
misc_RNA,12,45,23,2,1,0,2,85
rRNA,0,0,0,4,0,0,0,4
snRNA,7,23,17,2,0,0,1,50
snoRNA,2,3,3,28,0,0,0,36
tRNA,4,3,4,1,0,0,4,16


In [70]:
print("Full label breakdown:")
print(res["label"].value_counts(), "\n")
print("Positives by projection mode (span-only == divergent / SPAN-recovered):")
print(res.loc[res["class"] == "POSITIVE", "proj_mode"].value_counts(), "\n")
print("NAME_MATCH examples (near-gold ortholog pairs):")
res.loc[res.label == "NAME_MATCH",
        ["transcript_id","biotype","gene_name","evidence_mm","evidence_name","overlap_bp"]].head(10)


Full label breakdown:
label
GEOM_REJECT             3471
BIOTYPE_MATCH           1846
FAMILY_MISMATCH         1372
SPAN_CANDIDATE          1347
UNLABELED_INTERGENIC     416
NAME_MATCH               343
PSEUDO_HIT               311
NO_CHAINS                152
NO_PROJECTION            119
Name: count, dtype: int64 

Positives by projection mode (span-only == divergent / SPAN-recovered):
proj_mode
exon    2189
Name: count, dtype: int64 

NAME_MATCH examples (near-gold ortholog pairs):


,transcript_id,biotype,gene_name,evidence_mm,evidence_name,overlap_bp
341,ENST00000384048.1,snoRNA,SNORD37,ENSMUST00000082507.3,Snord37,61
1023,ENST00000386972.1,miRNA,MIR24-2,ENSMUST00000083607.3,Mir24-2,72
1024,ENST00000385073.1,miRNA,MIR27A,ENSMUST00000083510.3,Mir27a,79
1025,ENST00000385245.1,miRNA,MIR23A,ENSMUST00000083677.3,Mir23a,73
1034,ENST00000384881.1,miRNA,MIR181C,ENSMUST00000083549.4,Mir181c,89
1035,ENST00000384853.1,miRNA,MIR181D,ENSMUST00000102383.3,Mir181d,72
1040,ENST00000621445.1,miRNA,MIR1199,ENSMUST00000117015.3,Mir1199,118
1396,ENST00000384437.1,snoRNA,SNORA68,ENSMUST00000104375.3,Snora68,125
4211,ENST00000787383.1,lncRNA,LNCOB1,ENSMUST00000172483.3,Lncob1,5980
4212,ENST00000585408.2,lncRNA,LNCOB1,ENSMUST00000172483.3,Lncob1,5975


## 6. Free side-analyses: SPAN divergent-orthologs & pseudogene concordance
- **SPAN divergent-ortholog candidates**: exon projection failed but the *span* landed on a compatible
  mm39 ncRNA → the chain aligner missed a real ortholog. These are the cases to send through RiNALMo
  (does the unaligned syntenic window still embed near the reference?).
- **PSEUDO_HIT**: how often ncRNA loci project onto annotated mm39 pseudogenes — evidence for/against
  keeping a P_PGENES-style class for ncRNA.

In [71]:
# SPAN divergent-ortholog candidates: exons did not align, but the span projection is gene-length-sized
# (len_ratio in band) AND lands on a compatible mm39 ncRNA -> the chain aligner missed a real ortholog.
# These are the cases to send through RiNALMo (does the unaligned syntenic window embed near the reference?).
span_div = res[res["label"] == "SPAN_CANDIDATE"]
print(f"SPAN divergent-ortholog candidates (feed to RiNALMo): {len(span_div):,}")
print("  distinct mm39 genes:", span_div.evidence_mm.nunique(),
      "| distinct hg38 GENES:", span_div.transcript_id.map(lambda x: ref_gene.get(x)).nunique())
display(span_div[["transcript_id","biotype","gene_name","evidence_mm","evidence_name","len_ratio","cover_frac_mm"]].head(15))

print("\nPSEUDO_HIT by reference biotype:")
print(res.loc[res.label == "PSEUDO_HIT"].groupby("biotype")["transcript_id"].count())

SPAN divergent-ortholog candidates (feed to RiNALMo): 1,347
  distinct mm39 genes: 199 | distinct hg38 GENES: 283


,transcript_id,biotype,gene_name,evidence_mm,evidence_name,len_ratio,cover_frac_mm
34,ENST00000727640.1,lncRNA,NaN,ENSMUST00000318194.1,Gm13962,1.140,0.038
77,ENST00000797535.1,lncRNA,NaN,ENSMUST00000253483.1,Gm51808,1.673,0.375
102,ENST00000818438.1,lncRNA,NaN,ENSMUST00000269954.1,Gm40713,0.996,0.533
119,ENST00000718988.1,lncRNA,NaN,ENSMUST00000244149.2,Gm56961,0.947,1.000
120,ENST00000718987.1,lncRNA,NaN,ENSMUST00000244149.2,Gm56961,0.947,1.000
121,ENST00000718985.1,lncRNA,NaN,ENSMUST00000244149.2,Gm56961,0.947,1.000
122,ENST00000718986.1,lncRNA,NaN,ENSMUST00000244149.2,Gm56961,0.947,1.000
231,ENST00000844822.1,lncRNA,NaN,ENSMUST00000312377.1,Gm68752,0.804,0.056
232,ENST00000592934.2,lncRNA,NaN,ENSMUST00000312377.1,Gm68752,0.808,0.056
238,ENST00000590531.2,lncRNA,NaN,ENSMUST00000270338.1,Gm31057,0.659,0.085



PSEUDO_HIT by reference biotype:
biotype
lncRNA      308
miRNA         2
misc_RNA      1
Name: transcript_id, dtype: int64


## 7. (Optional) Score the current pipeline against this oracle
Point `PIPE` at a real **hg38-vs-mm39** rna_toga output (a table listing transcripts called ORTH).
`precision` and `recall_floor` are computed only over *decidable* rows (POSITIVE + CLEAN_NEGATIVE);
UNLABELED / NO_PROJECTION are excluded because absence of mm39 annotation is not a true negative.

In [72]:
PIPE = ROOT/"big_test/toga_mini_results/rna_orthologous_regions.tsv"  # <-- adjust to your hg38-vs-mm39 run

if not PIPE.exists():
    print(f"No pipeline table at {PIPE}.")
    print("Set PIPE to your hg38-vs-mm39 rna_toga ORTH output (needs a transcript_id column).")
else:
    pt = pd.read_csv(PIPE, sep="\t", dtype=str)
    orth_tx = set(pt["transcript_id"])                       # transcripts called ORTH to >=1 chain
    ev = res.copy()
    ev["pipe_orth"] = ev["transcript_id"].isin(orth_tx)
    dec = ev[ev["class"].isin(["POSITIVE", "CLEAN_NEGATIVE"])]
    tp = int(((dec["class"] == "POSITIVE")       & dec["pipe_orth"]).sum())
    fp = int(((dec["class"] == "CLEAN_NEGATIVE") & dec["pipe_orth"]).sum())
    fn = int(((dec["class"] == "POSITIVE")       & ~dec["pipe_orth"]).sum())
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    rec_floor = tp / (tp + fn) if (tp + fn) else float("nan")
    print(f"decidable={len(dec):,}  precision={prec:.3f}  recall_floor={rec_floor:.3f}")
    print(pd.crosstab(dec["class"], dec["pipe_orth"]))


decidable=3,561  precision=nan  recall_floor=0.000
pipe_orth       False
class                
CLEAN_NEGATIVE   1372
POSITIVE         2189


## 8. Annotation-free multi-species tree-consistency
Project each hg38 ncRNA **gene** through the chains you already have, in **each** species, and record
whether it lands cleanly (exon-mode, `len_ratio` in band). No query annotation needed -> works for
every species regardless of GENCODE quality (this is what sidesteps your "only a few species have
reasonable ncRNA annotation" problem).

A real ortholog shows a **nested** presence pattern down the tree (present in near species, dropping
out in farther ones); presence in a distant species while absent in a nearer one is a phylogenetic
**violation** (spurious / paralog / repeat). We then cross-check breadth + violations against the
mm39-annotation oracle from section 4 — two *independent* signals that should agree.

`SPECIES_RUN` defaults to a 3-species subset (rheMac10 / mm39 / monDom5) for a fast first pass; set it
to `None` to run the full 8-species panel (slower — each chain file is loaded once, then freed).

In [73]:
import gc

# Species panel ordered by approx divergence from human. Chains you already have on disk.
SPECIES_PANEL = [
    ("gorGor6",  "input_data/chains/hg38.gorGor6.allfilled.chain.gz",  1),
    ("rheMac10", "input_data/chains/hg38.rheMac10.allfilled.chain.gz", 2),
    ("mm39",     "input_data/chains/hg38.mm39.allfilled.chain.gz",     3),
    ("bosTau9",  "input_data/chains/hg38.bosTau9.allfilled.chain.gz",  4),
    ("equCab3",  "input_data/chains/hg38.equCab3.allfilled.chain.gz",  4),
    ("canFam4",  "input_data/chains/hg38.canFam4.allfilled.chain.gz",  4),
    ("susScr11", "input_data/chains/hg38.susScr11.allfilled.chain.gz", 4),
    ("monDom5",  "input_data/chains/hg38ToMonDom5.over.chain.gz",      5),
]
SPECIES_RUN = None   # None => whole panel; or a list of names for a fast subset
panel = [s for s in SPECIES_PANEL if (SPECIES_RUN is None or s[0] in SPECIES_RUN)]
cols  = [s[0] for s in panel]

# real topology (Fitch parsimony); hg38 is the always-present reference leaf
_apes   = ("hg38", "gorGor6")
_prim   = (_apes, "rheMac10")
_euarch = (_prim, "mm39")
_cetart = ("bosTau9", "susScr11")
_zoo    = ("equCab3", "canFam4")
_laura  = (_cetart, _zoo)
_boreo  = (_euarch, _laura)
TREE    = (_boreo, "monDom5")

def _prune(node, keep):
    if isinstance(node, str):
        return node if (node == "hg38" or node in keep) else None
    kids = [k for k in (_prune(c, keep) for c in node) if k is not None]
    if not kids: return None
    return kids[0] if len(kids) == 1 else tuple(kids)

def _fitch(node, pres):
    if isinstance(node, str):
        return ({1} if (node == "hg38" or pres.get(node, 0)) else {0}), 0
    sets, ch = [], 0
    for c in node:
        s, k = _fitch(c, pres); sets.append(s); ch += k
    inter = sets[0] & sets[1]
    return (inter, ch) if inter else (sets[0] | sets[1], ch + 1)

PRUNED = _prune(TREE, set(cols))

# gene-level representatives (longest transcript per gene) among the ncRNA targets
gene_rep = {}
for t in targets:
    g = ref_gene.get(t.id)
    if g is None:
        continue
    cur = gene_rep.get(g)
    if cur is None or (t.end - t.start) > (cur.end - cur.start):
        gene_rep[g] = t
print(f"ncRNA genes (reps): {len(gene_rep):,} on {list(chain_index)}")

def build_index(ch_sp, chroms):
    ci = {}
    for c in chroms:
        cl = ch_sp.get_by_target_chrom(c)
        if not cl:
            continue
        sp = np.array([ch.t_span for ch in cl], dtype=np.int64)
        sc = np.array([ch.score for ch in cl], dtype=np.int64)
        ci[c] = (sp[:, 0], sp[:, 1], sc, cl)
    return ci

def project_present(t, ci):
    idx = ci.get(t.chrom)
    if idx is None:
        return 0
    cs, ce, sc, cl = idx
    m = (cs < t.end) & (ce > t.start)
    if not m.any():
        return 0
    cand = np.where(m)[0]; cand = cand[np.argsort(-sc[cand])][:TOP_K_CHAINS]
    glen = max(1, t.end - t.start); lo, hi = LEN_RATIO_BAND
    for i in cand:
        region, mode = project_region(t, cl[int(i)])
        if region is None or mode != "exon":
            continue
        if lo <= (region[1] - region[0]) / glen <= hi:
            return 1
    return 0

chroms = list(chain_index.keys())
present = {}
for name, path, rank in panel:
    t0 = time.time()
    ch_sp = chains if name == "mm39" else read_chain_file(str(ROOT/path))
    ci = build_index(ch_sp, chroms)
    present[name] = {g: project_present(t, ci) for g, t in gene_rep.items()}
    print(f"{name:9s} rank{rank}  present {sum(present[name].values())}/{len(gene_rep)}  ({time.time()-t0:.0f}s)")
    if name != "mm39":
        del ch_sp, ci; gc.collect()

pm = pd.DataFrame({name: pd.Series(present[name]) for name in cols})
pm.index.name = "gene_id"
pm["n_present"] = pm[cols].sum(axis=1)
pm["changes"]   = pm.apply(lambda r: _fitch(PRUNED, {c: int(r[c]) for c in cols})[1], axis=1)

# cross-check against the mm39-annotation oracle (independent signals should agree)
res2 = res.copy(); res2["gene_id"] = res2.transcript_id.map(lambda x: ref_gene.get(x))
gene_class = res2.groupby("gene_id")["class"].agg(lambda s: s.value_counts().index[0])
pm = pm.join(gene_class.rename("anno_class"))

print("\nMean breadth / parsimony-changes by mm39-annotation class:")
print(pm.groupby("anno_class")[["n_present", "changes"]].mean().round(2))
print(f"\nBroadly conserved & coherent (present in >={len(cols)-1}/{len(cols)}, changes<=1) by class:")
print(pm[(pm.n_present >= len(cols) - 1) & (pm.changes <= 1)]["anno_class"].value_counts())
pm.head(10)

ncRNA genes (reps): 2,166 on ['chr19', 'chr21']
gorGor6   rank1  present 1714/2166  (24s)
rheMac10  rank2  present 1601/2166  (58s)
mm39      rank3  present 865/2166  (2s)
bosTau9   rank4  present 1252/2166  (183s)
equCab3   rank4  present 1418/2166  (82s)
canFam4   rank4  present 1269/2166  (234s)
susScr11  rank4  present 1320/2166  (128s)
monDom5   rank5  present 415/2166  (9s)

Mean breadth / parsimony-changes by mm39-annotation class:
                n_present  changes
anno_class                        
CLEAN_NEGATIVE       5.70     1.21
GEOM_REJECT          3.58     1.99
NO_PROJECTION        2.80     1.51
POSITIVE             6.63     1.18
PSEUDO_HIT           4.12     1.28
SPAN_CAND            2.51     1.86
UNLABELED            5.53     1.57

Broadly conserved & coherent (present in >=7/8, changes<=1) by class:
anno_class
POSITIVE          258
CLEAN_NEGATIVE    246
UNLABELED          60
GEOM_REJECT        32
PSEUDO_HIT          8
NO_PROJECTION       7
SPAN_CAND           3
Name: 

,gorGor6,rheMac10,mm39,bosTau9,equCab3,canFam4,susScr11,monDom5,n_present,changes,anno_class
gene_id,,,,,,,,,,,
ENSG00000292982.2,1,1,1,1,1,1,1,0,7,1,GEOM_REJECT
ENSG00000282807.4,1,1,0,1,1,1,1,0,6,2,GEOM_REJECT
ENSG00000283801.1,1,1,0,1,1,1,1,0,6,2,NO_PROJECTION
ENSG00000282591.2,1,1,1,1,1,1,0,0,6,2,UNLABELED
ENSG00000295826.1,0,0,0,0,0,0,0,0,0,1,GEOM_REJECT
ENSG00000290363.2,0,1,1,1,1,1,1,0,6,2,PSEUDO_HIT
ENSG00000295047.1,0,1,0,0,0,0,0,0,1,2,SPAN_CAND
ENSG00000282535.1,1,1,1,1,1,1,1,0,7,1,CLEAN_NEGATIVE
ENSG00000282508.3,0,0,0,0,0,0,0,0,0,1,GEOM_REJECT


## 9. Structured-ncRNA family oracle (Infernal cmscan vs Rfam) — independent of GENCODE
The third weak-supervision signal, scoped to structured RNAs (miRNA/snoRNA/snRNA/tRNA) where Rfam has
coverage. We extract each candidate's **hg38** sequence and its **projected mm39** locus, `cmscan` both
against `Rfam.cm`, and check the **family (Rfam accession) is concordant**. This is independent of both
GENCODE annotation quality and the mm39-annotation oracle, and it recovers miRNAs that fail chain
projection (the family is found directly). cmscan runs on the short extracted windows only (seconds),
not whole genomes; it is executed as a shell step and the notebook parses the `tblout`.

In [74]:
import os

# structured ncRNA biotypes where Rfam has coverage (lncRNA excluded on purpose)
STRUCT_BIOTYPES = ["miRNA", "snoRNA", "scaRNA", "snRNA", "tRNA"]
TMP = ROOT/"notebooks/_cmscan_tmp"; os.makedirs(TMP, exist_ok=True)
PAD = 50   # bp padding around projected locus so the full CM element is captured

struct = {g: t for g, t in gene_rep.items() if ref_biotype.get(t.id) in STRUCT_BIOTYPES}
print(f"structured genes on {list(chain_index)}: {len(struct)}")

def project_to_mm39(t):
    """Best exon-mode projection (in band) -> (q_chrom, q_start, q_end) or None. Uses mm39 `chains`."""
    idx = chain_index.get(t.chrom)
    if idx is None:
        return None
    cs, ce, sc, cl = idx
    m = (cs < t.end) & (ce > t.start)
    if not m.any():
        return None
    cand = np.where(m)[0]; cand = cand[np.argsort(-sc[cand])][:TOP_K_CHAINS]
    glen = max(1, t.end - t.start); lo, hi = LEN_RATIO_BAND
    for i in cand:
        ch = cl[int(i)]
        region, mode = project_region(t, ch)
        if region is None or mode != "exon":
            continue
        if lo <= (region[1] - region[0]) / glen <= hi:
            return ch.q_chrom, region[0], region[1]
    return None

hg_rows, mm_rows = [], []
for g, t in struct.items():
    hg_rows.append((t.chrom, max(0, t.start), t.end, g))
    pr = project_to_mm39(t)
    if pr:
        qc, qs, qe = pr
        mm_rows.append((qc, max(0, qs - PAD), qe + PAD, g))

pd.DataFrame(hg_rows).to_csv(TMP/"hg38_struct.bed", sep="\t", header=False, index=False)
pd.DataFrame(mm_rows).to_csv(TMP/"mm39_struct.bed", sep="\t", header=False, index=False)
print(f"hg38 regions: {len(hg_rows)} | mm39 projected regions: {len(mm_rows)}")
print("mm39 q_chrom values:", sorted({r[0] for r in mm_rows}))
print("struct biotype counts:", pd.Series([ref_biotype.get(t.id) for t in struct.values()]).value_counts().to_dict())

structured genes on ['chr19', 'chr21']: 272
hg38 regions: 272 | mm39 projected regions: 134
mm39 q_chrom values: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr2', 'chr3', 'chr5', 'chr7', 'chr8', 'chr9', 'chrM', 'chrX']
struct biotype counts: {'miRNA': 170, 'snRNA': 50, 'snoRNA': 36, 'tRNA': 16}


In [75]:
# Parse cmscan --fmt 2 tblout: best Rfam family (accession) per gene, then hg38<->mm39 concordance.
# Run the shell step first (twoBitToFa + cmscan) so hg38.tblout / mm39.tblout exist in TMP.
def parse_cmscan(path):
    fam = {}   # gene_id -> (rf_accession, family_name, evalue, score)
    if not os.path.isfile(path):
        print("missing:", path); return fam
    with open(path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.split()
            if len(f) < 19 or f[18] != "!":   # inc == '!' -> significant (GA-passing)
                continue
            gene, acc, family = f[3], f[2], f[1]
            try:
                ev = float(f[17])
            except ValueError:
                continue
            cur = fam.get(gene)
            if cur is None or ev < cur[2]:
                fam[gene] = (acc, family, ev, f[16])
    return fam

hg_fam = parse_cmscan(str(TMP/"hg38.tblout"))
mm_fam = parse_cmscan(str(TMP/"mm39.tblout"))
print(f"hg38 structured genes with an Rfam hit: {len(hg_fam)}/{len(struct)}")
print(f"mm39 projected loci with an Rfam hit:   {len(mm_fam)}/{len(mm_rows) if 'mm_rows' in dir() else '?'}")

both = set(hg_fam) & set(mm_fam)
conc = sum(1 for g in both if hg_fam[g][0] == mm_fam[g][0])
print(f"family on BOTH sides: {len(both)} | concordant Rfam accession: {conc} ({100*conc/max(1,len(both)):.0f}%)")

rows = []
for g, t in struct.items():
    hf, mf = hg_fam.get(g), mm_fam.get(g)
    rows.append(dict(gene_id=g, biotype=ref_biotype.get(t.id),
                     hg_family=hf[1] if hf else None, hg_acc=hf[0] if hf else None,
                     mm_family=mf[1] if mf else None, mm_acc=mf[0] if mf else None,
                     concordant=bool(hf and mf and hf[0] == mf[0])))
cm = pd.DataFrame(rows)
res2 = res.copy(); res2["gene_id"] = res2.transcript_id.map(lambda x: ref_gene.get(x))
gclass = res2.groupby("gene_id")["class"].agg(lambda s: s.value_counts().index[0])
cm = cm.merge(gclass.rename("anno_class"), left_on="gene_id", right_index=True, how="left")

print("\nRfam family-concordance (hg38==mm39) by mm39-annotation class:")
print(cm.groupby("anno_class")["concordant"].agg(["sum", "count"]))
print("\nhg38 Rfam-detection rate by biotype (recovers cases even when GENCODE/chain fail):")
print(cm.assign(hit=cm.hg_family.notna()).groupby("biotype")["hit"].agg(["sum", "count"]))
cm[cm.concordant].head(12)

hg38 structured genes with an Rfam hit: 180/272
mm39 projected loci with an Rfam hit:   54/134
family on BOTH sides: 52 | concordant Rfam accession: 51 (98%)

Rfam family-concordance (hg38==mm39) by mm39-annotation class:
                sum  count
anno_class                
CLEAN_NEGATIVE    2     73
GEOM_REJECT       0     48
NO_PROJECTION     0     83
POSITIVE         46     56
PSEUDO_HIT        0      2
UNLABELED         3     10

hg38 Rfam-detection rate by biotype (recovers cases even when GENCODE/chain fail):
         sum  count
biotype            
miRNA     87    170
snRNA     47     50
snoRNA    30     36
tRNA      16     16


,gene_id,biotype,hg_family,hg_acc,mm_family,mm_acc,concordant,anno_class
4,ENSG00000207507.1,snRNA,U6,RF00026,U6,RF00026,True,POSITIVE
5,ENSG00000207357.1,snRNA,U6,RF00026,U6,RF00026,True,POSITIVE
16,ENSG00000206775.1,snoRNA,SNORD37,RF00440,SNORD37,RF00440,True,POSITIVE
18,ENSG00000207630.1,miRNA,mir-7,RF00053,mir-7,RF00053,True,POSITIVE
26,ENSG00000200237.1,snoRNA,SNORA70,RF00156,SNORA70,RF00156,True,POSITIVE
28,ENSG00000209645.1,snoRNA,SNORD105,RF00584,SNORD105,RF00584,True,POSITIVE
29,ENSG00000238531.1,snoRNA,snoU105B,RF01173,snoU105B,RF01173,True,POSITIVE
36,ENSG00000207752.1,miRNA,mir-199,RF00144,mir-199,RF00144,True,POSITIVE
40,ENSG00000209702.1,snoRNA,SNORD41,RF00588,SNORD41,RF00588,True,POSITIVE
45,ENSG00000284387.1,miRNA,mir-24,RF00178,mir-24,RF00178,True,POSITIVE


## 10. Union: weak-supervision consensus + is the TOGA approach adequate for ncRNA?
Combine the three independent oracles into one **consensus label** per gene (ortholog / non-ortholog /
ambiguous / unlabeled), then ask the real question: **do TOGA-style features separate ortholog from
non-ortholog against this consensus, and do the ncRNA-specific features add anything?**

- `TOGA-like` = synteny + coverage (the original principle).
- `NEW` = locus-length ratio, chain multiplicity, multi-species conservation breadth + parsimony changes.

Interpretation guide: if TOGA-like already scores well -> the principle transfers; if `ALL` beats it
meaningfully -> ncRNA needs augmented features; per-biotype splits show whether one model suffices.
Caveat: coverage/len_ratio share the projection that defines the annotation label (partly circular);
synteny, multiplicity and the multi-species features are the independent evidence.

In [76]:
# ---- weak-supervision consensus of the 3 oracles (per gene) ----
# strong oracles = mm39 annotation + cmscan family; tree-consistency is supporting only.
# votes: +1 ortholog, -1 not-ortholog, 0 abstain.
res2 = res.copy(); res2["gene_id"] = res2.transcript_id.map(lambda x: ref_gene.get(x))
gclass = res2.groupby("gene_id")["class"].agg(lambda s: s.value_counts().index[0])
def anno_vote(c):
    return 1 if c == "POSITIVE" else (-1 if c in ("CLEAN_NEGATIVE", "PSEUDO_HIT") else 0)
cm_hg = {g: hg_fam[g][0] for g in hg_fam}; cm_mm = {g: mm_fam[g][0] for g in mm_fam}
def cm_vote(g):
    a, b = cm_hg.get(g), cm_mm.get(g)
    return 0 if not (a and b) else (1 if a == b else -1)
tcols = [c for c in pm.columns if c not in ("n_present", "changes", "anno_class")]
tree_pos = (pm["n_present"] >= max(2, len(tcols) // 2)) & (pm["changes"] <= 1)
rows = []
for g in gene_rep:
    av, cv = anno_vote(gclass.get(g)), cm_vote(g); tv = int(bool(tree_pos.get(g, False)))
    pos, neg = (av > 0) + (cv > 0), (av < 0) + (cv < 0)
    lab = "ORTHOLOG" if (pos and not neg) else ("NON_ORTHOLOG" if (neg and not pos) else ("AMBIGUOUS" if (pos and neg) else "UNLABELED"))
    rows.append((g, av, cv, tv, lab, pos + neg))
con = pd.DataFrame(rows, columns=["gene_id","anno_vote","cm_vote","tree_vote","consensus","n_strong"]).set_index("gene_id")
print("consensus label distribution:"); print(con.consensus.value_counts())
both = con[(con.anno_vote != 0) & (con.cm_vote != 0)]
print(f"\nanno vs cmscan agree where both vote: {(np.sign(both.anno_vote) == np.sign(both.cm_vote)).sum()}/{len(both)}")
con.head()

consensus label distribution:
consensus
UNLABELED       1295
NON_ORTHOLOG     480
ORTHOLOG         388
AMBIGUOUS          3
Name: count, dtype: int64

anno vs cmscan agree where both vote: 46/49


,anno_vote,cm_vote,tree_vote,consensus,n_strong
gene_id,,,,,
ENSG00000292982.2,0,0,1,UNLABELED,0
ENSG00000282807.4,0,0,0,UNLABELED,0
ENSG00000283801.1,0,0,0,UNLABELED,0
ENSG00000282591.2,0,0,0,UNLABELED,0
ENSG00000295826.1,0,0,0,UNLABELED,0


In [77]:
# ---- Does the TOGA-style feature set separate ortholog from non-ortholog (vs the consensus)? ----
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# fast SYNTENY proxy: number of distinct genes the gene's top chain spans (TOGA's flagship feature)
gspan = {}
for c in chain_index:
    d = {}
    for t in ref.get_by_chrom(c):
        g = ref_gene.get(t.id)
        if g is None:
            continue
        s, e = d.get(g, (t.start, t.end)); d[g] = (min(s, t.start), max(e, t.end))
    arr = np.array([[s, e] for s, e in d.values()], dtype=np.int64); gspan[c] = (arr[:, 0], arr[:, 1])

def synteny_of(t):
    idx = chain_index.get(t.chrom)
    if idx is None:
        return 0
    cs, ce, sc, cl = idx; m = (cs < t.end) & (ce > t.start)
    if not m.any():
        return 0
    ch = cl[int(np.where(m)[0][np.argmax(sc[m])])]
    ts, te = int(ch.t_span[0]), int(ch.t_span[1]); gs, ge = gspan[t.chrom]
    return int(((gs < te) & (ge > ts)).sum())

rbt = res.set_index("transcript_id")
lab = con[con.consensus.isin(["ORTHOLOG", "NON_ORTHOLOG"])]
rows = []
for g, r in lab.iterrows():
    t = gene_rep[g]; rr = rbt.loc[t.id] if t.id in rbt.index else None
    gv = lambda k: float(rr[k]) if (rr is not None and pd.notna(rr[k])) else 0.0
    rows.append(dict(gene_id=g, y=int(r.consensus == "ORTHOLOG"), biotype=ref_biotype.get(t.id),
        synteny=synteny_of(t), cover_frac_region=gv("cover_frac_region"), cover_frac_mm=gv("cover_frac_mm"),
        len_ratio=gv("len_ratio"), n_cand_chains=gv("n_cand_chains"),
        n_present=float(pm.n_present.get(g, 0)), changes=float(pm.changes.get(g, 0)), gene_len=gv("gene_len")))
X = pd.DataFrame(rows).set_index("gene_id")
print("labeled genes:", len(X), "| ortholog:", int(X.y.sum()))
print(X.groupby("biotype").y.agg(["sum", "count"]))

feat_all = ["synteny","cover_frac_region","cover_frac_mm","len_ratio","n_cand_chains","n_present","changes","gene_len"]
TOGA = ["synteny","cover_frac_region","cover_frac_mm"]        # TOGA-style: synteny + coverage
NEW  = ["len_ratio","n_cand_chains","n_present","changes"]    # ncRNA-specific additions
cv = StratifiedKFold(5, shuffle=True, random_state=0)
def auc(feats, clf):
    Z = StandardScaler().fit_transform(X[feats].values)
    return cross_val_score(clf, Z, X.y.values, cv=cv, scoring="roc_auc").mean()
print("\n5-fold CV ROC-AUC vs consensus label:")
for name, feats in [("ALL", feat_all), ("TOGA-like", TOGA), ("NEW-only", NEW)]:
    print(f"  {name:10s} logreg={auc(feats, LogisticRegression(max_iter=1000)):.3f}  GBM={auc(feats, GradientBoostingClassifier()):.3f}")
lr = LogisticRegression(max_iter=1000).fit(StandardScaler().fit_transform(X[feat_all]), X.y)
print("\nstandardized logreg coefficients (+=ortholog):")
print(pd.Series(lr.coef_[0], index=feat_all).sort_values().round(2).to_string())
print("\nNOTE: cover_frac_* / len_ratio are computed on the same projection that partly defines the\nannotation label -> partly circular. synteny, n_cand_chains, n_present, changes are independent.")

labeled genes: 868 | ortholog: 388
          sum  count
biotype             
lncRNA    324    718
miRNA      24     86
misc_RNA    2     15
rRNA        4      4
snRNA       2      9
snoRNA     28     30
tRNA        4      6

5-fold CV ROC-AUC vs consensus label:
  ALL        logreg=0.784  GBM=0.815
  TOGA-like  logreg=0.719  GBM=0.755
  NEW-only   logreg=0.625  GBM=0.629

standardized logreg coefficients (+=ortholog):
len_ratio           -0.29
n_cand_chains       -0.17
cover_frac_region   -0.08
synteny              0.24
gene_len             0.37
changes              0.55
cover_frac_mm        0.83
n_present            1.18

NOTE: cover_frac_* / len_ratio are computed on the same projection that partly defines the
annotation label -> partly circular. synteny, n_cand_chains, n_present, changes are independent.


## Notes / caveats
- **Positives are trustworthy; absence is not.** mm39 lncRNA annotation is deep (152k) but
  structured-ncRNA is thin (2.2k miRNA, 1.5k snoRNA), so `UNLABELED`/`NO_PROJECTION` for a snoRNA is
  probably an annotation gap, not a loss. Treat this as positive-unlabeled, not positive-negative.
- **Length cap** (`LEN_CAP_COEF`) is your guard against matching a tiny gene to a huge locus; sweep it.
- **`gl_exo`/`exon_perc` are CDS-based** in `rna_toga.py` and degenerate for ncRNA — once you have this
  truth set, re-fit the classifier on features computed against exon *blocks*, and ablate per biotype.
- Next: add reciprocity (needs the mm39→hg38 chain or a block-swap) only if `NAME_MATCH` precision looks noisy.
